<a href="https://colab.research.google.com/github/mohamed-hossam1/RAG/blob/main/%D9%8CRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
    "fastapi>=0.139.0" \
    "openai>=2.45.0" \
    "python-dotenv>=1.2.2" \
    "uvicorn>=0.51.0" \
    "qdrant-client>=1.13.0" \
    "FlagEmbedding>=1.4.0"

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
from openai import OpenAI

In [ ]:
!wget -O data.jsonl "https://raw.githubusercontent.com/mohamed-hossam1/RAG/main/data/meta_Electronics_with_category_ratings_100_sample_1000.jsonl"

In [ ]:
df_items = pd.read_json(
    "data.jsonl",
    lines=True
)


df_items.head()

In [ ]:
list(df_items["features"].items())[0]

In [ ]:
list(df_items["images"].items())[0]

In [ ]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [ ]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")


In [ ]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [ ]:
list(df_items["description"].items())[0]

In [ ]:
list(df_items["image"].items())[0]

In [ ]:
df_sample = df_items.sample(300, random_state=42)

len(df_sample)

In [ ]:
data_to_embed = df_sample[
    [
        "description",
        "image",
        "rating_number",
        "price",
        "average_rating",
        "parent_asin",
    ]
].to_dict(orient="records")

data_to_embed

In [ ]:
from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel( "BAAI/bge-m3", use_fp16=False )

In [ ]:
def get_embedding(text):
  output = model.encode(text, max_length=8192)
  return output["dense_vecs"].tolist()

In [ ]:
response = get_embedding("Mohamed")

len(response)
print(len(response))
print(response)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq docker.io

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.recreate_collection(
    collection_name="Amazon-items-collection-00",
    vectors_config=VectorParams(
        size=len(get_embedding("test")),
        distance=Distance.COSINE,
    ),
)

print("Collection 'Amazon-items-collection-00' created successfully!")

In [ ]:
pointstructs = []

for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data["description"])

    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data,
        )
    )

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
len(pointstructs)

In [ ]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    wait=True,
    points=pointstructs,
)

In [ ]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)

    results = client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )

    return results

query_text = "I need a protector for my iPad"
search_results = retrieve_data(query_text)

for hit in search_results.points:
    print(f"ID: {hit.id}, Score: {hit.score:.4f}")
    print(f"Title: {hit.payload.get('description', '')[:100]}...")
    print("-" * 30)